# Notebook 9 -- Conclusion générale et perspectives
### Ce que la recherche a établi, et comment elle a permis de mieux poser la question restante

> ****Note de provenance.**** Les notebooks de recherche originaux seront présents dans le répertoire du projet, accompagnés des données et des fichiers nécessaires aux expériences. Le présent notebook fait uniquement office de **notebook d'analyse et de synthèse** : il ne remplace pas les notebooks originaux et ne constitue pas une nouvelle implémentation des expérimentations.
>
> Le code, les résultats et les sorties présentés dans ce notebook sont analysés à partir des travaux réalisés dans les notebooks originaux. Aucun résultat n'a été recalculé ni inventé : lorsqu'une information n'était pas déterminable à partir des notebooks ou des données disponibles, cela est signalé explicitement plutôt que comblé. Ce notebook n'est donc pas conçu pour être exécuté isolément de haut en bas ; il a pour objectif de **documenter, structurer, comparer et interpréter les résultats déjà obtenus**, ainsi que de mettre en évidence la progression scientifique entre les différentes approches étudiées.


## Objectif de ce notebook

Ce notebook conclut le mémoire. Il ne cherche pas à désigner une méthode gagnante -- aucune des
sept méthodes testées dans les notebooks 2 à 7 ne résout entièrement le problème posé au
notebook 1. Sa fonction est différente : montrer que cette recherche, plutôt que d'avoir simplement
essayé plusieurs algorithmes, a progressivement permis de **décomposer** le problème initial en deux
sous-problèmes de nature différente, d'identifier précisément où se situent les limites des
approches statistiques testées, et d'en tirer deux perspectives de recherche qui découlent
directement de ces limites plutôt que d'être ajoutées après coup.

---

## 1. Retour sur la problématique

Le notebook 1 posait la question suivante : comment reconstruire ou prédire de manière fiable le
signal vibratoire d'un capteur momentanément indisponible, à partir des informations fournies par
les autres capteurs présents sur la machine -- en particulier lorsque cette indisponibilité survient
pendant un événement vibratoire important ? Une distinction y était déjà annoncée comme devant
structurer toute l'analyse : reproduire la forme générale d'un signal et reproduire son amplitude
exacte sont deux exigences différentes. L'ensemble des notebooks qui ont suivi peut se relire comme
une mise à l'épreuve progressive de cette distinction.

## 2. Synthèse des approches étudiées

Trois familles conceptuelles ont été explorées, chacune motivée par la limite de la précédente :

- Une **relation statique** entre capteurs (corrélation, régression linéaire, PCA -- notebook 2),
  qui teste si une redondance instantanée suffit.
- Une **relation dynamique**, se voulant physique ou quasi physique (Dictionary Learning au
  notebook 3, puis SINDy au notebook 4, combiné à un observateur d'état SSI + filtre de Kalman
  étendu au notebook 5), qui teste si une dynamique identifiable explique mieux les observations
  qu'une simple corrélation.
- Une **relation apprise par apprentissage supervisé** (XGBoost, avec ou sans représentations SSI,
  SINDy ou CNN 1D -- notebooks 6 et 7), qui teste si un modèle non contraint par une forme
  explicite fait mieux que les deux approches précédentes.

Le notebook 8 a mis ces sept méthodes en regard sur une grille de lecture commune (nature de
l'information exploitée, caractère statistique ou physique, capacité à prédire l'amplitude, à
reproduire les pics, interprétabilité, robustesse).

## 3. Enseignements principaux

Plusieurs constats, établis au fil des notebooks 2 à 8, ressortent avec une robustesse suffisante
pour être considérés comme les enseignements principaux de ce travail :

- Une redondance statistique réelle existe entre le capteur cible et plusieurs autres capteurs de
  la machine (corrélations Train de 0,29 à 0,85, notebook 1), suffisante pour qu'une régression
  linéaire simple obtienne un résultat positif sur une fenêtre de panne courte (R² = 0,234,
  notebook 2).
- Un état latent linéaire de dimension 8 s'identifie correctement par SSI et reste observable même
  sans le capteur cible (notebook 5) -- mais une dynamique continue non linéaire de cet état
  **n'a pas pu être identifiée** par SINDy, de façon reproductible sur deux bibliothèques de
  fonctions différentes (notebook 4).
- Sur une évaluation instant par instant portant sur l'ensemble du Test, plusieurs méthodes
  atteignent des corrélations élevées avec le signal réel (jusqu'à 0,928 pour certaines variantes
  XGBoost, notebook 6) -- mais ces mêmes méthodes, réévaluées sur un véritable scénario de panne
  continue, voient leur R² s'effondrer jusqu'à -38,5 (notebook 7). Ce contraste, obtenu sur la
  même famille de modèles avec seulement un changement de protocole d'évaluation, est le résultat
  le mieux instrumenté de tout le corpus.

## 4. Dynamique et forme *versus* amplitude

Ces constats convergent vers une même lecture, qu'il convient de formuler avec la prudence
nécessaire plutôt que comme une loi générale :

> Les résultats obtenus au cours de cette étude suggèrent que les relations statistiques exploitées
> entre les différents capteurs permettent principalement de retrouver certaines caractéristiques
> de la dynamique et de la forme du signal manquant, tandis que la reconstruction fidèle de son
> amplitude reste beaucoup plus difficile.

Ce n'est pas la même chose que d'affirmer, de façon absolue, que « la corrélation statistique ne
permet pas de retrouver l'amplitude » -- une telle formulation serait plus générale que ce que ce
corpus démontre. Ce que les notebooks 2 à 7 montrent précisément, c'est que, **sur les données et
les scénarios testés**, aucune des sept méthodes ne reconstruit fidèlement l'amplitude pendant une
fenêtre de panne continue, alors que plusieurs d'entre elles reconstruisent une forme ou une
tendance temporelle exploitable dans des conditions d'évaluation moins exigeantes.

Trois éléments viennent nuancer et expliquer, sans le démontrer expérimentalement, ce constat :
les capteurs disponibles peuvent être fortement corrélés au capteur cible ; cette corrélation peut
effectivement permettre de retrouver une dynamique ou une tendance commune ; mais une corrélation
élevée, calculée sur un régime de fonctionnement donné, ne garantit pas une connaissance précise de
l'amplitude réelle du signal, en particulier pendant un régime différent ou un événement
transitoire. L'amplitude dépend vraisemblablement, entre autres, de la propagation des vibrations
dans la structure, de la position relative des capteurs, de la structure mécanique de la machine et
des propriétés physiques des matériaux traversés. **Aucun de ces éléments n'est démontré
expérimentalement par les notebooks de ce corpus** : ce sont des hypothèses explicatives et des
pistes de recherche, pas des résultats établis -- elles sont reprises et développées à la section 7
de ce notebook.

Cette distinction constitue l'un des principaux enseignements de ce travail, et peut se reformuler
comme la décomposition du problème initial en deux sous-problèmes de nature différente :

**Problème A -- reconstruire la dynamique et la forme du signal perdu.** Les résultats de ce
corpus montrent que les approches étudiées peuvent apporter une information utile sur cette
composante, à des degrés divers selon la méthode et le protocole d'évaluation.

**Problème B -- reconstruire son amplitude physique réelle.** Ce problème reste ouvert. Il semble,
au vu des résultats obtenus, nécessiter une compréhension plus approfondie de la propagation
physique des vibrations dans la machine -- une compréhension qu'aucune des méthodes testées ici,
qu'elle soit statistique ou d'apprentissage supervisé, n'a intégrée.

## 5. Limites de l'approche actuelle

Avant de proposer des perspectives, il faut rappeler ce que ce corpus ne permet pas de conclure
(détaillé notebook par notebook aux notebooks 1 à 8) : l'absence de modélisation physique explicite
de l'atténuation ou de la propagation dans toutes les méthodes testées ; la rareté des événements
transitoires disponibles pour évaluer, et *a fortiori* pour apprendre, une reconstruction
d'amplitude (4 pics détectés sur l'ensemble du jeu de données du notebook 1) ; l'absence d'un
protocole d'évaluation strictement harmonisé entre la piste physique (notebooks 4-5) et la piste
supervisée (notebooks 6-7), qui reposent sur deux jeux de données légèrement différents ; et,
plus largement, le fait qu'aucune méthode testée ne construit ni ne vérifie une relation de
transfert mécanique causale entre les points de mesure -- toutes reposent sur des corrélations ou
des relations apprises, dont la validité en dehors du régime observé en apprentissage n'est pas
garantie.

---

## 6. Première perspective : reconstruction et détection d'anomalies

Le constat de la section 4 n'implique pas nécessairement que l'effort de reconstruction soit sans
valeur pratique. Même si la reconstruction ne permet pas de retrouver avec suffisamment de
précision l'amplitude absolue du signal manquant, la forme et la dynamique prédites -- lorsqu'elles
sont exploitables, comme le suggèrent certains résultats des notebooks 2, 5 et 6 -- peuvent malgré
tout contenir une information utile pour la surveillance de la machine. L'idée, qui découle
directement des résultats plutôt que d'être une piste indépendante, est la suivante :

```text
Capteur indisponible
        |
        v
Utilisation des autres capteurs
        |
        v
Prediction / reconstruction de la dynamique
        |
        v
Obtention d'un signal estime
        |
        v
Analyse de la region reconstruite
        |
        v
Detection d'une incoherence / anomalie
        |
        v
Indice de suspicion d'un evenement anormal
```

Cette approche changerait légèrement l'objectif poursuivi jusqu'ici. Il ne s'agirait plus
nécessairement de *remplacer parfaitement le signal du capteur manquant*, mais d'*exploiter le
signal reconstruit comme une information indirecte permettant de détecter qu'un événement
inhabituel a probablement eu lieu pendant la période où le capteur était indisponible*.

### Proposition d'algorithme -- une perspective de recherche, pas une méthode validée

Il est important de souligner que rien dans les notebooks 1 à 8 ne met en œuvre ni ne valide une
telle détection d'anomalies : elle est présentée ici uniquement comme une perspective de recherche
qui découle logiquement des résultats obtenus. L'idée serait d'appliquer un algorithme de
détection d'anomalies sur la région correspondant au gap de données, selon un schéma du type :

```text
Donnees disponibles avant le gap
              +
Autres capteurs pendant le gap
              |
              v
       Modele de reconstruction
              |
              v
         Signal estime
              |
              v
       Analyse de l'anomalie
              |
              v
     Score / indice d'anomalie
```

Un tel algorithme pourrait rechercher, sur le signal reconstruit : une rupture inhabituelle de
dynamique, une variation brutale, une forme incompatible avec le comportement habituel de la
machine, une évolution temporelle anormale, ou un comportement différent des régimes de
fonctionnement précédemment observés. Le résultat serait un score d'anomalie indiquant qu'un
événement inhabituel *pourrait* avoir eu lieu pendant la période de perte de données -- une
piste de recherche consisterait à évaluer si la reconstruction de la dynamique peut être exploitée
comme support à une détection d'anomalies, et non une affirmation que cette méthode permet de
détecter les anomalies.

### Mise en garde sur l'interprétation

Un score d'anomalie, même élevé, ne permettrait pas de conclure automatiquement que *la machine est
en panne*. Il indiquerait seulement que *le comportement reconstruit présente des caractéristiques
inhabituelles compatibles avec un événement anormal*. Une validation physique, ou une information
provenant d'autres sources (maintenance, autres capteurs, journal d'exploitation), resterait
nécessaire avant de conclure qu'un choc ou une vibration anormale a réellement eu lieu pendant
cette période. Cette distinction est importante pour conserver, sur cette perspective comme sur le
reste du mémoire, une interprétation scientifique rigoureuse : un indice de suspicion n'est pas un
diagnostic.

## 7. Deuxième perspective : caractérisation expérimentale de la propagation

La section 6 propose d'exploiter ce que la reconstruction *sait déjà faire* (la dynamique). Il reste
la question ouverte à la section 4 : comment récupérer l'information d'amplitude qui semble
manquer dans une approche essentiellement statistique ? Les résultats obtenus au cours de cette
étude suggèrent que cette question ne peut probablement pas être abordée uniquement en améliorant,
successivement, les modèles statistiques ou d'apprentissage automatique testés dans les notebooks
2 à 7 -- trois familles de méthodes conceptuellement différentes ont déjà été essayées sans résoudre
le problème de l'amplitude. Il pourrait être nécessaire d'introduire davantage de connaissance
physique du système plutôt que de continuer à raffiner des modèles purement fondés sur les données.

Une étude expérimentale de la propagation des vibrations entre les différents emplacements de
capteurs constituerait une piste naturelle. Une telle étude pourrait chercher à caractériser :
l'atténuation de l'amplitude entre un point d'excitation et un point de mesure ; la propagation des
vibrations dans la structure ; la réponse de différents matériaux traversés ; l'influence de la
distance entre capteurs ; l'influence de leur position et de leur orientation ; les effets propres
à la structure mécanique de la machine ; les éventuelles différences de comportement selon la
fréquence de l'excitation ; et les phénomènes de transmission et d'atténuation associés. L'objectif
d'une telle étude serait de déterminer expérimentalement comment une excitation mesurée à un
endroit de la machine se retrouve, en forme et en amplitude, à un autre endroit -- une information
qu'aucune des méthodes testées dans ce corpus ne possède, puisque toutes apprennent uniquement à
partir de corrélations observées, sans connaissance du mécanisme physique sous-jacent.

## 8. Vers un modèle physique de transfert / atténuation

Une étude expérimentale de ce type pourrait ensuite servir à construire ou calibrer un modèle
physique de propagation, selon la logique suivante :

```text
Etude experimentale
        |
        v
Mesure de la propagation des vibrations
        |
        v
Caracterisation de l'attenuation
        |
        v
Identification des parametres physiques
        |
        v
Modele de propagation / transfert
        |
        v
Combinaison avec les modeles statistiques
        |
        v
Meilleure estimation de l'amplitude
```

Cette approche est présentée ici comme une perspective de recherche, et non comme une solution déjà
démontrée : rien dans ce corpus ne permet d'anticiper la précision qu'un tel modèle physique
atteindrait, ni la difficulté de son identification expérimentale sur une machine industrielle
réelle plutôt qu'en conditions contrôlées.

### Lien avec la matrice de transfert H

Cette perspective permet de reboucler avec l'intuition initiale de ce mémoire (notebook 2) : une
matrice, ou plus généralement une fonction, de transfert H reliant les capteurs entre eux. Les
notebooks 2 et 3 ont montré qu'aucune des matrices obtenues jusqu'ici (coefficients de régression,
composantes PCA, dictionnaire de motifs, matrices A et C de SSI) ne constitue une véritable matrice
de transfert physique -- toutes restent des objets statistiques appris sur les données observées.
Une étude expérimentale de la propagation, telle que décrite ci-dessus, pourrait permettre de
revenir à cette intuition initiale, mais avec une base physique et expérimentale plus solide qu'une
simple corrélation. Il faut cependant se garder d'annoncer que le modèle qui en résulterait serait
nécessairement une matrice H classique, au sens strict d'une relation linéaire invariante : cette
perspective pourrait conduire à une représentation de type fonction ou matrice de transfert, dont
les paramètres seraient informés ou calibrés expérimentalement -- une formulation volontairement
prudente, qui montre l'évolution de la réflexion menée dans ce mémoire sans en anticiper le
résultat.

## 9. Vers une approche hybride physique + données

La perspective la plus cohérente avec l'ensemble de ce travail n'est donc probablement pas de
remplacer les méthodes statistiques et d'apprentissage par un modèle physique, ni l'inverse, mais
de les combiner. Les méthodes testées dans les notebooks 2 à 7 exploitent efficacement la
redondance et les motifs statistiques disponibles dans les données pour reconstruire une dynamique
ou une forme ; un modèle physique de propagation, tel qu'esquissé à la section 8, apporterait une
information complémentaire, absente de toutes les méthodes testées ici, sur l'échelle et
l'atténuation réelles du signal. La combinaison des deux -- une dynamique apprise sur les données et
une échelle physique calibrée expérimentalement -- constitue une perspective à long terme plutôt
qu'une prochaine étape immédiatement réalisable avec les seuls éléments de ce corpus.

---

## 10. Conclusion générale

Cette recherche a suivi une progression qui peut se relire comme une boucle logique, où chaque
étape a été motivée par la limite de la précédente, et où la dernière perspective proposée ici est
une conséquence naturelle de l'ensemble du parcours plutôt qu'une idée ajoutée en fin de mémoire :

```text
Correlations entre capteurs
        |
        v
Reconstruction statistique (regression, PCA, dictionnaire)
        |
        v
Recherche d'une dynamique physique identifiable
        |
        v
SINDy
        |
        v
Combinaisons SINDy + SSI + EKF / Kalman
        |
        v
Approches supervisees (XGBoost, CNN)
        |
        v
Constat : forme / dynamique mieux accessible que l'amplitude
        |
        v
Nouvelle possibilite : exploiter la reconstruction pour detecter une anomalie
        |
        v
Question ouverte : comment reconstruire reellement l'amplitude ?
        |
        v
Perspective : etude experimentale de la propagation des vibrations
        |
        v
Modele physique de transfert / attenuation
        |
        v
Combinaison physique + donnees
```

Ce mémoire ne désigne pas de méthode gagnante, et ce n'était pas son objectif. Sa contribution
tient dans la démonstration reproductible, sur sept méthodes de trois familles conceptuelles
différentes, d'une limite structurelle commune, et dans la décomposition qu'elle a permise du
problème initial en deux sous-problèmes de nature différente (section 4) : la reconstruction de la
dynamique et de la forme d'un signal vibratoire à partir d'autres capteurs est, dans une certaine
mesure, accessible ; la reconstruction fidèle de son amplitude pendant un événement critique ne
l'est pas, avec les données et les méthodes disponibles à ce jour. La recherche a apporté des
réponses, mais elle a également permis de mieux définir la question restante -- et c'est
précisément cette limite, mieux comprise, qui justifie les deux perspectives proposées ci-dessus.

## 11. Perspectives de recherche

En résumé, deux perspectives se dégagent directement des résultats de ce corpus, sans prétendre à
une validation qu'aucun notebook ne fournit :

- **Court terme.** Évaluer si la reconstruction de la dynamique, telle qu'obtenue par les méthodes
  des notebooks 2, 5 et 6, peut servir de support à une détection d'anomalies sur la fenêtre de
  panne -- une piste de recherche à instruire, pas une méthode déjà démontrée.
- **Long terme.** Caractériser expérimentalement les mécanismes de propagation et d'atténuation des
  vibrations dans la machine, en vue de construire ou calibrer un modèle physique susceptible
  d'améliorer la reconstruction de l'amplitude, puis de le combiner aux approches statistiques et
  d'apprentissage déjà développées dans ce travail.

### Message scientifique final

La reconstruction d'un capteur manquant ne doit peut-être pas être considérée uniquement comme un
problème de prédiction de signal. Les résultats obtenus suggèrent qu'une partie de l'information
dynamique peut être reconstruite à partir des autres capteurs, et potentiellement exploitée pour
détecter une anomalie dans le signal prédit. Ainsi, même si le signal réel n'est pas directement
disponible pendant la période de perte de données, l'analyse du signal reconstruit pourrait
permettre d'identifier un comportement inhabituel et de déclencher une alerte, plutôt que de ne
disposer d'aucune information. En revanche, la reconstruction fidèle de l'amplitude semble
nécessiter une compréhension plus approfondie des mécanismes physiques de propagation et
d'atténuation des vibrations dans la machine.

À plus long terme, la combinaison suivante se dégage comme perspective de recherche : données des
capteurs, modèles dynamiques, détection d'anomalies, et connaissance physique de la propagation des
vibrations. Ce mémoire n'apporte pas cette combinaison -- il en établit, à travers les limites
identifiées notebook après notebook, la nécessité.